# 04 - Route every request: auto-resolve vs human assistance

For each incoming request decide: can it be **resolved automatically** (we have sufficient historical precedent to reuse) or does it **need a human**? Uses three signals:
1. **Intent + classifier confidence** (from notebook 03)
2. **Precedent** - is this intent common enough in history that patterns exist (`intent_prevalence`)
3. **Author history** - volume of prior mentions, resolved-reply rate, and typical reply latency (computed from the full raw `twcs.csv`)

In [148]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

# Rule constants (tunable, documented in the routing report)
CONF_LO = 0.30         # below classifier confidence -> ambiguous -> human
PREV_MIN = 0.02        # intent must be >=2% of history to have precedent
HIST_VOL_MIN = 3       # author needs this many prior inbound tweets
HIST_REPLY_MIN = 0.3   # ...and a resolved-reply rate above this
AUTO_INTENTS = {'order_status', 'email_contact'}  # routine, template-able
AUTO_REPLY_MIN = {'history_replied': 2, 'history_reply_rate': 0.7}
t0 = time.time()


In [149]:
# resolve classified requests: attach the 03 output, /working, or local fallback
CAND_CLASSIFIED = ['/kaggle/input/amazon-help-classified/amazon_help_classified.csv',
                  '/kaggle/working/amazon_help_classified.csv',
                  '../data/processed/amazon_help_classified.csv']
CLASSIFIED_PATH = next((p for p in CAND_CLASSIFIED if os.path.exists(p)), CAND_CLASSIFIED[-1])

df = pd.read_csv(CLASSIFIED_PATH)
print(f"Requests loaded: {len(df)} from {CLASSIFIED_PATH}")
print(df["pred_intent"].value_counts().to_dict())
df.head()


Requests loaded: 31862 from /kaggle/working/amazon_help_classified.csv
{'customer_service': 12439, 'delivery_issue': 6633, 'email_contact': 4381, 'appreciation': 4293, 'order_status': 4116}


,tweet_id,author_id,created_at,text,intent,low_confidence,topic_confidence,pred_intent,pred_confidence
0,173478,156612,Fri Nov 24 22:57:58 +0000 2017,yes i have i ask it to resend the code a few time,customer_service,False,0.029544,email_contact,0.661466
1,341491,197580,Sat Oct 28 07:07:02 +0000 2017,what type of communication r u do with me day ...,customer_service,False,0.029118,customer_service,0.742502
2,870162,326613,Fri Oct 13 16:13:49 +0000 2017,will u let people know what action i have take...,customer_service,False,0.023888,customer_service,0.936374
3,2576953,254045,Sun Dec 03 04:02:46 +0000 2017,and when i try to contact you i get a message ...,order_status,False,0.051908,order_status,0.744373
4,407048,212017,Tue Oct 10 07:23:17 +0000 2017,give i have already spend several hour sort yo...,customer_service,False,0.019227,customer_service,0.526089


## 1. Author history from the full dataset
Every author gets: `history_volume` (prior inbound mentions to @AmazonHelp), `history_replied` (how many got a reply), `history_reply_rate`, `avg_reply_hours` (median-ish latency to a resolution reply). `history_sufficient` = enough volume AND a decent reply rate.

In [157]:
!ls /kaggle/input/datasets/thoughtvector/

In [158]:
import kagglehub
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/thoughtvector/customer-support-on-twitter


In [159]:
# resolve raw history: public Kaggle twcs dataset, or local copy
CAND_RAW = ['/kaggle/input/datasets/thoughtvector/customer-support-on-twitter/twcs/twcs.csv',
           '/kaggle/input/twcs/twcs.csv',
           '/kaggle/working/twcs.csv',
           '../data/raw/twcs.csv']
RAW_PATH = next((p for p in CAND_RAW if os.path.exists(p)), CAND_RAW[-1])

print("Building author history from raw twcs...")
raw = pd.read_csv(RAW_PATH, dtype={"tweet_id": str, "author_id": str,
                                   "response_tweet_id": str, "in_response_to_tweet_id": str})
print(f"Raw rows: {len(raw):,}")

amz_mask = raw["inbound"] == 1
mention = raw.loc[amz_mask, "text"].str.lower().str.contains("amazonhelp", na=False)
amz_in = raw.loc[amz_mask & mention].copy()
print(f"AmazonHelp inbound mentions: {len(amz_in):,}")

responded = amz_in[amz_in["response_tweet_id"].notna()]
print(f"...of which got a reply: {len(responded):,}")

# reply latency: map each replied mention to its outbound response tweet
out = raw[raw["inbound"] == 0][["tweet_id", "created_at"]]
lat = responded[["tweet_id", "author_id", "created_at", "response_tweet_id"]].merge(
    out, left_on="response_tweet_id", right_on="tweet_id", suffixes=("_in", "_out"))
lat["wait_h"] = (pd.to_datetime(lat["created_at_out"]) - pd.to_datetime(lat["created_at_in"])).dt.total_seconds() / 3600
lat = lat[lat["wait_h"].between(0, 72)]

hist = amz_in.groupby("author_id").agg(
    history_volume=("tweet_id", "count"),
    history_replied=("response_tweet_id", lambda s: s.notna().sum()),
).reset_index()
hist["history_reply_rate"] = (hist["history_replied"] / hist["history_volume"]).clip(upper=1.0)
avg_wait = lat.groupby("author_id")["wait_h"].mean().rename("avg_reply_hours")
hist = hist.merge(avg_wait, on="author_id", how="left")
hist["history_sufficient"] = ((hist["history_volume"] >= HIST_VOL_MIN) &
                              (hist["history_reply_rate"] >= HIST_REPLY_MIN))
print(f"Authors with history: {len(hist):,}")


Building author history from raw twcs...
Raw rows: 2,811,774
AmazonHelp inbound mentions: 135,171
...of which got a reply: 100,198
Authors with history: 48,137


In [160]:
df["author_id"] = df["author_id"].astype(str)
df = df.merge(hist, on="author_id", how="left")
for c in ["history_volume", "history_replied"]:
    df[c] = df[c].fillna(0).astype(int)
df["history_reply_rate"] = df["history_reply_rate"].fillna(0.0)
df["avg_reply_hours"] = df["avg_reply_hours"].fillna(np.nan)
df["history_sufficient"] = df["history_sufficient"].fillna(False)
print(df[["history_volume", "history_replied", "history_reply_rate", "history_sufficient"]].describe().loc[["mean", "50%"]])


      history_volume  history_replied  history_reply_rate
mean        8.660003         6.115592            0.736120
50%         4.000000         3.000000            0.760952


## 2. Intent precedent
`intent_prevalence` = share of all requests that share this classified intent. A low-prevalence intent means no template exists yet -> human.

In [161]:
N = len(df)
prev_share = df["pred_intent"].value_counts(normalize=True)
df["intent_prevalence"] = df["pred_intent"].map(prev_share)
df["has_precedent"] = df["intent_prevalence"] >= PREV_MIN
print("Intent prevalence:"); print((prev_share * 100).round(1).to_dict())


Intent prevalence:
{'customer_service': 39.0, 'delivery_issue': 20.8, 'email_contact': 13.7, 'appreciation': 13.5, 'order_status': 12.9}


## 3. Routing rule (explainable, no black box)

| Condition | Route | Reason |
|---|---|---|
| `pred_intent == appreciation` | auto | acknowledge & close |
| `pred_confidence < 0.30` | assist | low prediction confidence |
| `low_confidence` (NMF silver label weak) | assist | unclear intent |
| no precedent (prevalence < 2%) | assist | novel intent |
| history not sufficient | assist | insufficient history |
| routine intent (order_status / email_contact) | auto | template exists |
| consistent resolver (>=2 replied, rate >= 0.7) | auto | reuse prior resolution |
| otherwise | assist | needs judgment |

In [162]:
def route_request(row):
    if row["pred_intent"] == "appreciation":
        return "auto", "acknowledge_and_close"
    if row["pred_confidence"] < CONF_LO:
        return "assist", "low_prediction_confidence"
    if row["low_confidence"]:
        return "assist", "unclear_intent"
    if not row["has_precedent"]:
        return "assist", "novel_intent_no_precedent"
    if not row["history_sufficient"]:
        return "assist", "insufficient_history"
    if row["pred_intent"] in AUTO_INTENTS:
        return "auto", "routine_intent_history"
    if (row["history_replied"] >= AUTO_REPLY_MIN["history_replied"] and
            row["history_reply_rate"] >= AUTO_REPLY_MIN["history_reply_rate"]):
        return "auto", "consistent_resolution_history"
    return "assist", "needs_judgment"

routs = df.apply(route_request, axis=1)
df["route"] = [r[0] for r in routs]
df["route_reason"] = [r[1] for r in routs]

print("=== Route distribution ===")
print(df["route"].value_counts())
print(); print("Routes by intent:"); print(pd.crosstab(df["pred_intent"], df["route"]))
print(); print("Reasons:"); print(df["route_reason"].value_counts())


=== Route distribution ===
route
assist    16647
auto      15215
Name: count, dtype: int64

Routes by intent:
route             assist  auto
pred_intent                   
appreciation           0  4293
customer_service    8832  3607
delivery_issue      4647  1986
email_contact       1590  2791
order_status        1578  2538

Reasons:
route_reason
insufficient_history             6473
unclear_intent                   5870
consistent_resolution_history    5593
routine_intent_history           5329
acknowledge_and_close            4293
needs_judgment                   4165
low_prediction_confidence         139
Name: count, dtype: int64


In [163]:
auto = df[df["route"] == "auto"]
assist = df[df["route"] == "assist"]
print(f"Auto:        {len(auto):,}  avg conf={auto["pred_confidence"].mean():.3f}  avg history={auto["history_volume"].mean():.1f}")
print(f"Human assist:{len(assist):,}  avg conf={assist["pred_confidence"].mean():.3f}  avg history={assist["history_volume"].mean():.1f}")
print(); print("Sample AUTO (template-able):")
print(auto[["text", "pred_intent", "route_reason", "history_volume", "history_reply_rate"]].head(6).to_string(index=False))
print(); print("Sample ASSIST (human needed):")
print(assist[["text", "pred_intent", "route_reason", "history_volume", "history_reply_rate"]].head(6).to_string(index=False))


Auto:        15,215  avg conf=0.717  avg history=10.5
Human assist:16,647  avg conf=0.691  avg history=7.0

Sample AUTO (template-able):
                                                                                                                                         text      pred_intent                  route_reason  history_volume  history_reply_rate
             and when i try to contact you i get a message that you be not able to find my order and the call do not connect to your employee     order_status        routine_intent_history               6            0.333333
                  just now amazon have change it to delivery tomorrow this game cost extra money to buy the deluxe and get it day early count   delivery_issue consistent_resolution_history               6            0.833333
          already contactedthey be say it be technical glitchneed to wait for few hour the start only you be fail to deliver recharge service   delivery_issue consistent_resolution_history

## 4. Sensitivity - how much does the global auto rate move with the confidence gate?
The choice of `CONF_LO` is the main operational lever; the split should be robust to it.

In [164]:
def route_request_with_conf(row, conf_lo):
    if row["pred_intent"] == "appreciation":
        return "auto", "acknowledge_and_close"
    if row["pred_confidence"] < conf_lo:
        return "assist", "low_prediction_confidence"
    if row["low_confidence"]:
        return "assist", "unclear_intent"
    if not row["has_precedent"]:
        return "assist", "novel_intent_no_precedent"
    if not row["history_sufficient"]:
        return "assist", "insufficient_history"
    if row["pred_intent"] in AUTO_INTENTS:
        return "auto", "routine_intent_history"
    if (row["history_replied"] >= AUTO_REPLY_MIN["history_replied"] and
            row["history_reply_rate"] >= AUTO_REPLY_MIN["history_reply_rate"]):
        return "auto", "consistent_resolution_history"
    return "assist", "needs_judgment"

print("Sensitivity (auto-rate as CONF_LO varies):")
sens = {}
for c in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    rr = df.apply(lambda r: route_request_with_conf(r, c), axis=1)
    sens[str(c)] = round(float(np.mean([x[0] == "auto" for x in rr])), 4)
    print(f"  CONF_LO={c:.2f}  auto_rate={sens[str(c)]:.1%}")


Sensitivity (auto-rate as CONF_LO varies):
  CONF_LO=0.15  auto_rate=47.9%
  CONF_LO=0.20  auto_rate=47.9%
  CONF_LO=0.25  auto_rate=47.9%
  CONF_LO=0.30  auto_rate=47.8%
  CONF_LO=0.35  auto_rate=47.2%
  CONF_LO=0.40  auto_rate=46.1%
  CONF_LO=0.50  auto_rate=42.6%


## 5. Save routed data + report + plot

In [165]:
# on Kaggle write to /kaggle/working so outputs become other notebooks' inputs
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '../data/processed'


In [166]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
colors = {"auto": "#2e7d32", "assist": "#c62828"}
big = df.groupby(["pred_intent", "route"]).size().unstack(fill_value=0)
big_pct = big.div(big.sum(axis=1), axis=0)
bottom = np.zeros(len(big_pct))
for r in ["auto", "assist"]:
    if r in big_pct:
        axes[0].bar(big_pct.index, big_pct[r], bottom=bottom, label=r, color=colors[r], alpha=0.9)
        bottom += big_pct[r]
axes[0].set_title("Auto-resolve share by intent")
axes[0].set_ylabel("share")
axes[0].set_xticklabels(big_pct.index, rotation=35, ha="right", fontsize=8)
axes[0].legend()

auto = df[df["route"] == "auto"]
assist = df[df["route"] == "assist"]
axes[1].pie([len(auto), len(assist)], labels=[f"auto\n{len(auto)}", f"assist\n{len(assist)}"],
            colors=[colors["auto"], colors["assist"]], startangle=90, autopct="%1.1f%%")
axes[1].set_title("Overall routing split")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/routing_plot.png")
plt.show()
print(f"Saved: {OUT_DIR}/routing_plot.png")

df.to_csv(f"{OUT_DIR}/amazon_help_routed.csv", index=False)
print(f"Saved: {OUT_DIR}/amazon_help_routed.csv ({len(df)} rows)")


Saved: /kaggle/working/routing_plot.png
Saved: /kaggle/working/amazon_help_routed.csv (31862 rows)


In [167]:
report = {
    "rule": {
        "appreciation": "auto (acknowledge & close)",
        "pred_confidence < conf_lo": "assist (low prediction confidence)",
        "low_confidence": "assist (unclear intent)",
        "intent_prevalence < prev_min": "assist (no precedent)",
        "history_sufficient == False": "assist (insufficient history)",
        "routine intent": "auto (order_status / email_contact)",
        "consistent resolver": "auto (>=2 replied, rate>=0.7)",
        "else": "assist (needs judgment)",
    },
    "constants": {
        "conf_lo": CONF_LO, "prev_min": PREV_MIN,
        "hist_vol_min": HIST_VOL_MIN, "hist_reply_min": HIST_REPLY_MIN,
        "auto_intents": sorted(AUTO_INTENTS), "auto_reply_min": AUTO_REPLY_MIN,
    },
    "routing": df["route"].value_counts().to_dict(),
    "routing_by_intent": {i: r.to_dict() for i, r in
                          df.groupby(["pred_intent", "route"]).size().unstack(fill_value=0).iterrows()},
    "reasons": df["route_reason"].value_counts().to_dict(),
    "auto_rate": round(float((df["route"] == "auto").mean()), 4),
    "avg_confidence_by_route": df.groupby("route")["pred_confidence"].mean().round(4).to_dict(),
    "avg_history_by_route": df.groupby("route")["history_volume"].mean().round(2).to_dict(),
    "sensitivity_conf_lo": sens,
    "authors": int(df["author_id"].nunique()),
}
with open(f"{OUT_DIR}/routing_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(f"Saved: {OUT_DIR}/routing_report.json")


Saved: /kaggle/working/routing_report.json


## 6. Build the evaluation golden set (N=200 silver + hand subset)

Stratify the classified corpus into a fixed 200-tweet evaluation set (same seed
as the notebooks), re-derive the *ideal* route from the GOLD intent + the
author's observed history, and attach the 50-tweet hand-labeled subset. These
files are consumed by `eval.py`.


In [ ]:
N_GOLDEN = 200
rng = np.random.RandomState(SEED)
pick = []
for intent, grp in df.groupby("intent"):                     # silver intent (02)
    n = max(1, int(round(N_GOLDEN * len(grp) / len(df))))
    pick.append(grp.sample(n=n, random_state=SEED))
golden = pd.concat(pick).sample(frac=1.0, random_state=SEED).reset_index(drop=True).head(N_GOLDEN).copy()


def golden_route(intent, row):
    """'Ideal' route a support lead picks, from the GOLD intent + author history.
    Mirrors route_request() but consumes gold_intent instead of pred_intent."""
    if intent == "appreciation":
        return "auto", "acknowledge_and_close"
    if not row["history_sufficient"]:
        return "assist", "insufficient_history"
    if intent in AUTO_INTENTS:
        return "auto", "routine_intent_history"
    if (row["history_replied"] >= AUTO_REPLY_MIN["history_replied"] and
            row["history_reply_rate"] >= AUTO_REPLY_MIN["history_reply_rate"]):
        return "auto", "consistent_resolution_history"
    return "assist", "needs_judgment"


golden["gold_intent"] = golden["intent"]              # silver label as gold
rr = golden.apply(lambda r: golden_route(r["gold_intent"], r), axis=1)
golden["gold_route"] = [x[0] for x in rr]
golden["gold_route_reason"] = [x[1] for x in rr]

keep = ["tweet_id", "author_id", "created_at", "text",
        "gold_intent", "gold_route", "gold_route_reason",
        "pred_intent", "pred_confidence", "topic_confidence",
        "history_volume", "history_replied", "history_reply_rate", "history_sufficient"]
golden = golden[keep]
assert len(golden) == N_GOLDEN
golden.to_csv(f"{OUT_DIR}/golden_eval.csv", index=False)
print(f"Saved: {OUT_DIR}/golden_eval.csv ({len(golden)} rows)")
print("route split:", golden["gold_route"].value_counts().to_dict())
print("intent split:", golden["gold_intent"].value_counts().to_dict())

# ---- hand-labeled 50-tweet subset (human review; attached for eval.py) ----
HAND_LABELS = {
    "2455253": ("other", "assist"),        # gibberish
    "515270": ("delivery_issue", "assist"),
    "757718": ("order_status", "assist"),  # schedule pickup (return)
    "1299400": ("customer_service", "assist"),  # vague complaint, mislabeled appreciation
    "348221": ("customer_service", "assist"),   # escalating complaint
    "374157": ("customer_service", "assist"),   # truncated thread filler
    "1303634": ("customer_service", "assist"),  # generic plea
    "288034": ("delivery_issue", "assist"),     # receipt/delivery discrepancy
    "1653422": ("customer_service", "assist"),  # vague follow-up
    "962520": ("customer_service", "auto"),     # echo-dot setup -> templated how-to
    "959294": ("customer_service", "assist"),
    "324244": ("customer_service", "assist"),   # prime escalation
    "2514674": ("customer_service", "assist"),  # loss compensation
    "280148": ("customer_service", "assist"),   # account lock complaint
    "226339": ("customer_service", "assist"),   # consumer-court threat
    "1072550": ("order_status", "assist"),      # stolen-money refund
    "2883304": ("email_contact", "assist"),     # no phone contact option
    "2346213": ("order_status", "assist"),      # return/courier dispute
    "2355189": ("email_contact", "assist"),     # fake email
    "2078539": ("other", "assist"),             # Spanish, off-topic
    "714749": ("delivery_issue", "assist"),
    "222687": ("delivery_issue", "auto"),       # carrier feedback -> ack
    "1449155": ("order_status", "auto"),        # cancel Prime -> templated
    "2440783": ("delivery_issue", "assist"),
    "1498542": ("delivery_issue", "assist"),
    "93672": ("delivery_issue", "assist"),
    "1229471": ("delivery_issue", "assist"),    # prime-now availability
    "233785": ("delivery_issue", "assist"),     # tracking mismatch
    "2400339": ("appreciation", "auto"),        # praise of delivery
    "1219174": ("delivery_issue", "assist"),
    "1983560": ("email_contact", "assist"),     # email-verification expiry
    "2184547": ("customer_service", "assist"),  # account security concern
    "2302234": ("order_status", "assist"),      # refund wrong amount
    "665640": ("delivery_issue", "assist"),
    "779871": ("delivery_issue", "assist"),
    "1570890": ("email_contact", "auto"),       # providing email id -> ack
    "1100564": ("email_contact", "assist"),     # sharing details, follow-up needed
    "1721485": ("customer_service", "assist"),
    "1732078": ("email_contact", "assist"),     # can't login to contact
    "1305186": ("delivery_issue", "assist"),
    "574928": ("order_status", "assist"),       # no ship date yet
    "785125": ("order_status", "assist"),       # gift must arrive before oct
    "1490808": ("order_status", "assist"),      # unresolved oct order
    "1843616": ("order_status", "assist"),      # cancel all pending orders
    "1302268": ("order_status", "assist"),      # refunded + product with me
    "2526919": ("order_status", "assist"),      # expedite shipment
    "2171450": ("order_status", "assist"),      # confirm vs cancel mismatch
    "467390": ("email_contact", "assist"),      # share detail thread
    "1445305": ("order_status", "auto"),        # cancel audible book -> templated
    "2619642": ("order_status", "assist"),      # next-day shipping failed
}

hand = golden[golden["tweet_id"].isin(HAND_LABELS)].set_index("tweet_id").copy()
assert not hand.index.duplicated().any(), "duplicate tweet_ids in hand pool"
for tid, (intent, route) in HAND_LABELS.items():
    assert tid in hand.index, f"hand label {tid} missing from golden_eval"
    hand.loc[tid, "hand_intent"] = intent
    hand.loc[tid, "hand_route"] = route
hand = hand.reset_index()
hand = hand[["tweet_id", "author_id", "created_at", "text",
             "gold_intent", "gold_route", "hand_intent", "hand_route",
             "history_volume", "history_replied", "history_reply_rate",
             "history_sufficient"]]
hand.to_csv(f"{OUT_DIR}/golden_hand.csv", index=False)
print(f"Saved: {OUT_DIR}/golden_hand.csv ({len(hand)} rows)")
print("hand intent:", hand["hand_intent"].value_counts().to_dict())
print("silver==hand intent agreement:", round(float((hand["gold_intent"] == hand["hand_intent"]).mean()), 3))
print("silver==hand route agreement:", round(float((hand["gold_route"] == hand["hand_route"]).mean()), 3))
